# COMM117 Week 4 Workshop — Optimisation Stability & Manual Data Parallel
**Duration:** 1 hour  
**GPU:** optional (runs on CPU)  

## Focus this week
- Effective batch size (gradient accumulation)
- LR warmup + cosine decay (schedule retuning)
- Gradient clipping
- Manual data-parallel gradient averaging (no `torch.distributed`)




## Part 0 — Setup (do not edit)
We provide:
1) a tiny character dataset
2) Workshop-4-style functional Transformer block code
3) a minimal tiny decoder-only model that *calls those functions*

You will implement the **training loop** + **manual DP** (the focus of this week).


In [1]:

import math, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---- tiny text corpus ----
CORPUS = (
    "ROMEO: But, soft! what light through yonder window breaks?\n"
    "It is the east, and Juliet is the sun.\n"
    "JULIET: O Romeo, Romeo! wherefore art thou Romeo?\n"
    "ROMEO: Shall I hear more, or shall I speak at this?\n"
    "JULIET: My bounty is as boundless as the sea.\n"
) * 40

alphabet = sorted(set(CORPUS))
to_id = {ch:i for i,ch in enumerate(alphabet)}
to_ch = {i:ch for ch,i in to_id.items()}
VOCAB = len(alphabet)

def tok_encode(s: str) -> torch.Tensor:
    return torch.tensor([to_id[c] for c in s], dtype=torch.long)

def tok_decode(ids: torch.Tensor) -> str:
    return "".join(to_ch[int(i)] for i in ids)

seq = tok_encode(CORPUS)
cut = int(0.9 * len(seq))
train_seq, val_seq = seq[:cut], seq[cut:]

def make_batch(split: str, bsz: int, ctx: int):
    d = train_seq if split == "train" else val_seq
    idx = torch.randint(0, len(d) - ctx - 1, (bsz,))
    x = torch.stack([d[i:i+ctx] for i in idx]).to(DEVICE)
    y = torch.stack([d[i+1:i+ctx+1] for i in idx]).to(DEVICE)
    return x, y

# ============================================================
# Workshop-4-style functional Transformer block definitions
# ============================================================

def causal_mask(N: int, device: str):
    # boolean mask broadcastable to (B,H,N,N) where True means "allowed"
    m = torch.tril(torch.ones(N, N, device=device)).bool()
    return m.view(1, 1, N, N)

def layer_norm(x, w, b, eps=1e-5):
    # x: (B,N,D), w/b: (D,)
    mu = x.mean(dim=-1, keepdim=True)
    var = ((x - mu)**2).mean(dim=-1, keepdim=True)
    xhat = (x - mu) / torch.sqrt(var + eps)
    return xhat * w + b

def sdpa(q, k, v, mask=None):
    # q,k,v: (B,H,N,Dh), mask: (1,1,N,N) bool
    B,H,N,Dh = q.shape
    att = (q @ k.transpose(-2, -1)) / math.sqrt(Dh)   # (B,H,N,N)
    if mask is not None:
        att = att.masked_fill(~mask, float("-inf"))
    att = F.softmax(att, dim=-1)
    out = att @ v                                     # (B,H,N,Dh)
    return out, att

def mha_packed_qkv(x, Wqkv, Wo, H, mask=None):
    # x: (B,N,D), Wqkv: (D,3D), Wo:(D,D)
    B,N,D = x.shape
    Dh = D // H
    qkv = x @ Wqkv                                   # (B,N,3D)
    q, k, v = qkv.split(D, dim=-1)
    q = q.view(B, N, H, Dh).transpose(1, 2)           # (B,H,N,Dh)
    k = k.view(B, N, H, Dh).transpose(1, 2)
    v = v.view(B, N, H, Dh).transpose(1, 2)
    out, att = sdpa(q, k, v, mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, N, D)
    y = out @ Wo                                      # (B,N,D)
    return y, att

def ffn(x, W1, b1, W2, b2):
    # x: (B,N,D), W1:(D,dff), b1:(dff), W2:(dff,D), b2:(D)
    h = x @ W1 + b1
    h = F.gelu(h)
    y = h @ W2 + b2
    return y

def preln_block(x, ln1_w, ln1_b, ln2_w, ln2_b,
                Wqkv, Wo, H,
                W1, b1, W2, b2,
                mask=None, eps=1e-5):
    # Pre-LN: x + MHA(LN(x)); x + FFN(LN(x))
    z = layer_norm(x, ln1_w, ln1_b, eps)
    attn_out, _ = mha_packed_qkv(z, Wqkv, Wo, H, mask=mask)
    x = x + attn_out
    z2 = layer_norm(x, ln2_w, ln2_b, eps)
    mlp_out = ffn(z2, W1, b1, W2, b2)
    x = x + mlp_out
    return x




DEVICE: cpu


## Part 1 — Baseline training 
Implement a **stable** baseline training loop.




### TODO: Write the forward() function for the decoder-only model

In [2]:
# ============================================================
# Tiny decoder-only model that uses the above functional blocks
# ============================================================

class TinyDecoderFunctional(nn.Module):
    def __init__(self, vocab, ctx=64, d=64, h=4, layers=2, dff_mult=4):
        super().__init__()
        assert d % h == 0
        self.ctx = ctx
        self.d = d
        self.h = h
        self.vocab = vocab

        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.zeros(1, ctx, d))
        nn.init.normal_(self.pos, std=0.02)

        dff = dff_mult * d

        # per-layer parameters (all nn.Parameter so AdamW can optimize)
        self.ln1_w = nn.ParameterList([nn.Parameter(torch.ones(d)) for _ in range(layers)])
        self.ln1_b = nn.ParameterList([nn.Parameter(torch.zeros(d)) for _ in range(layers)])
        self.ln2_w = nn.ParameterList([nn.Parameter(torch.ones(d)) for _ in range(layers)])
        self.ln2_b = nn.ParameterList([nn.Parameter(torch.zeros(d)) for _ in range(layers)])

        self.Wqkv = nn.ParameterList([nn.Parameter(torch.empty(d, 3*d)) for _ in range(layers)])
        self.Wo   = nn.ParameterList([nn.Parameter(torch.empty(d, d)) for _ in range(layers)])
        self.W1   = nn.ParameterList([nn.Parameter(torch.empty(d, dff)) for _ in range(layers)])
        self.b1   = nn.ParameterList([nn.Parameter(torch.zeros(dff)) for _ in range(layers)])
        self.W2   = nn.ParameterList([nn.Parameter(torch.empty(dff, d)) for _ in range(layers)])
        self.b2   = nn.ParameterList([nn.Parameter(torch.zeros(d)) for _ in range(layers)])

        for i in range(layers):
            nn.init.normal_(self.Wqkv[i], std=0.02)
            nn.init.normal_(self.Wo[i], std=0.02)
            nn.init.normal_(self.W1[i], std=0.02)
            nn.init.normal_(self.W2[i], std=0.02)

        self.lnf_w = nn.Parameter(torch.ones(d))
        self.lnf_b = nn.Parameter(torch.zeros(d))
        self.head = nn.Linear(d, vocab, bias=False)

    def forward(self, x, y=None):
        B, T = x.shape
        assert T <= self.ctx

        # token + position embeddings
        z = self.tok(x) + self.pos[:, :T, :]

        # causal mask
        mask = causal_mask(T, z.device)

        # transformer blocks
        for i in range(len(self.Wqkv)):
            z = preln_block(
                z,
                self.ln1_w[i], self.ln1_b[i],
                self.ln2_w[i], self.ln2_b[i],
                self.Wqkv[i], self.Wo[i], self.h,
                self.W1[i], self.b1[i], self.W2[i], self.b2[i],
                mask=mask,
            )

        # final norm + head
        z = layer_norm(z, self.lnf_w, self.lnf_b)
        logits = self.head(z)

        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        return logits, loss

    @torch.no_grad()
    def sample(self, prompt: str, new_tokens=120, temp=0.9):
        self.eval()
        ids = tok_encode(prompt).unsqueeze(0).to(DEVICE)
        for _ in range(new_tokens):
            ids_cond = ids[:, -self.ctx:]
            logits, _ = self(ids_cond)
            nxt = torch.multinomial(F.softmax(logits[:, -1, :]/max(temp,1e-6), dim=-1), 1)
            ids = torch.cat([ids, nxt], dim=1)
        self.train()
        return tok_decode(ids[0].cpu())

@torch.no_grad()
def eval_loss(model, ctx, iters=20, bsz=32):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        vals = []
        for _ in range(iters):
            xb, yb = make_batch(split, bsz, ctx)
            _, loss = model(xb, yb)
            vals.append(loss.item())
        out[split] = sum(vals) / len(vals)
    model.train()
    return out


In [3]:

# Baseline hyperparameters (CPU-friendly)
CTX = 64
D = 64
H = 4
LAYERS = 2

MICRO_BSZ = 16
ACCUM_STEPS = 2
MAX_STEPS = 120
BASE_LR = 3e-3
WARMUP = 20
CLIP_NORM = 1.0

print("Effective batch size =", MICRO_BSZ * ACCUM_STEPS)

model = TinyDecoderFunctional(VOCAB, ctx=CTX, d=D, h=H, layers=LAYERS).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=0.01)

# LR schedule: warmup then cosine decay to 0.
def lr_at(step: int) -> float:
    if step < WARMUP:
        return BASE_LR * (step + 1) / max(1, WARMUP)
    prog = (step - WARMUP) / max(1, (MAX_STEPS - WARMUP))
    prog = min(1.0, max(0.0, prog))
    return 0.5 * BASE_LR * (1.0 + math.cos(math.pi * prog))

# One training step with accumulation + clipping.
def train_step(step: int) -> float:
    lr = lr_at(step)
    for g in opt.param_groups:
        g["lr"] = lr

    opt.zero_grad(set_to_none=True)
    losses = []
    for _ in range(ACCUM_STEPS):
        xb, yb = make_batch("train", MICRO_BSZ, CTX)
        _, loss = model(xb, yb)
        (loss / ACCUM_STEPS).backward()
        losses.append(loss.item())

    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
    opt.step()
    return sum(losses) / len(losses)

print("Before:", eval_loss(model, CTX))

t0 = time.time()
for step in range(MAX_STEPS):
    last = train_step(step)
    if (step + 1) % 30 == 0:
        ev = eval_loss(model, CTX, iters=10)
        print(f"step {step+1:3d} | lr {lr_at(step):.2e} | train {ev['train']:.3f} | val {ev['val']:.3f} | last {last:.3f} | {time.time()-t0:.1f}s")

print("After:", eval_loss(model, CTX))

print("\n--- sample ---")
print(model.sample("ROMEO:", new_tokens=120, temp=0.9))


Effective batch size = 32
Before: {'train': 3.7307181000709533, 'val': 3.7340397953987123}
step  30 | lr 2.94e-03 | train 1.709 | val 1.704 | last 1.726 | 2.6s
step  60 | lr 2.01e-03 | train 1.105 | val 1.099 | last 1.086 | 4.9s
step  90 | lr 6.57e-04 | train 0.768 | val 0.753 | last 0.748 | 7.4s
step 120 | lr 7.40e-07 | train 0.690 | val 0.689 | last 0.701 | 9.9s
After: {'train': 0.697140896320343, 'val': 0.6962005734443665}

--- sample ---
ROMEO: But, w wighalindligh MET: JULIET: indleshespe JULIty t, ety JULIET: ty t bounty as iss s s thess and s Buthe s JULIthe 


### Baseline sanity check
Run after Part 1.

In [4]:

#  This should pass once your baseline is stable.
fresh = TinyDecoderFunctional(VOCAB, ctx=CTX, d=D, h=H, layers=LAYERS).to(DEVICE)
fresh_loss = eval_loss(fresh, CTX, iters=20)
trained_loss = eval_loss(model, CTX, iters=20)

print("fresh :", fresh_loss)
print("trained:", trained_loss)

assert trained_loss["val"] < fresh_loss["val"] - 0.05, "Not enough improvement. Retune LR/warmup/clip/accum."
print("Baseline looks stable (better than fresh init).")


fresh : {'train': 3.8629464626312258, 'val': 3.860857105255127}
trained: {'train': 0.6916763335466385, 'val': 0.7010882407426834}
Baseline looks stable (better than fresh init).


## Part 2 — Make loss *weird* (then diagnose)
Make an unstable run by changing schedule/clip/accum. Record symptoms.


In [ ]:

# Intentionally unstable configuration (expect spikes / poor training)
BAD_LR = 1e-1
BAD_WARMUP = 0
BAD_CLIP = 0.0
BAD_ACCUM = 1
STEPS = 60

bad = TinyDecoderFunctional(VOCAB, ctx=CTX, d=D, h=H, layers=LAYERS).to(DEVICE)
bad_opt = torch.optim.AdamW(bad.parameters(), lr=BAD_LR, weight_decay=0.01)

def bad_lr_at(step: int) -> float:
    # poorly tuned: constant large LR, no warmup
    return BAD_LR

for step in range(STEPS):
    lr = bad_lr_at(step)
    for g in bad_opt.param_groups:
        g["lr"] = lr

    bad_opt.zero_grad(set_to_none=True)

    # no accumulation
    xb, yb = make_batch("train", MICRO_BSZ, CTX)
    _, loss = bad(xb, yb)
    loss.backward()

    # no clipping
    if BAD_CLIP and BAD_CLIP > 0:
        torch.nn.utils.clip_grad_norm_(bad.parameters(), BAD_CLIP)

    bad_opt.step()

    if (step + 1) % 10 == 0:
        print(f"[bad] step {step+1:2d} | lr {lr:.1e} | loss {loss.item():.3f}")


## Part 3 — Manual Data Parallel (DP) gradient averaging
Simulate 2-worker DP: compute grads on two microbatches, average, step, and match the single-batch update.


In [ ]:

# Manual DP gradient averaging (2-worker simulation)

def copy_model(m):
    m2 = TinyDecoderFunctional(VOCAB, ctx=CTX, d=D, h=H, layers=LAYERS).to(DEVICE)
    m2.load_state_dict(m.state_dict())
    return m2

LR = 1e-3
FULL_B = 32

init = TinyDecoderFunctional(VOCAB, ctx=CTX, d=D, h=H, layers=LAYERS).to(DEVICE)
A = copy_model(init)   # single batch update
B = copy_model(init)   # dp update

x, y = make_batch("train", FULL_B, CTX)
x0, y0 = x[:FULL_B//2], y[:FULL_B//2]
x1, y1 = x[FULL_B//2:], y[FULL_B//2:]

# Path A: single batch
optA = torch.optim.SGD(A.parameters(), lr=LR)
optA.zero_grad(set_to_none=True)
_, lossA = A(x, y)
lossA.backward()
optA.step()

# Path B: two microbatches -> average grads
optB = torch.optim.SGD(B.parameters(), lr=LR)

# worker 0 grads
optB.zero_grad(set_to_none=True)
_, loss0 = B(x0, y0)
loss0.backward()
grads0 = [p.grad.detach().clone() for p in B.parameters()]

# worker 1 grads
optB.zero_grad(set_to_none=True)
_, loss1 = B(x1, y1)
loss1.backward()
grads1 = [p.grad.detach().clone() for p in B.parameters()]

# "all-reduce" average and apply
optB.zero_grad(set_to_none=True)
for p, g0, g1 in zip(B.parameters(), grads0, grads1):
    p.grad = (g0 + g1) / 2.0
optB.step()

# Compare parameters
max_diff = 0.0
for pA, pB in zip(A.parameters(), B.parameters()):
    max_diff = max(max_diff, (pA - pB).abs().max().item())
print("lossA(single) =", lossA.item(), "| avg micro loss =", (loss0.item() + loss1.item()) / 2.0)
print("max_diff:", max_diff)

assert max_diff < 1e-6, "DP averaging should match single-batch update (within tolerance)."
print(" Manual DP averaging matches single-batch update.")
